# F26-13 Phase 5: second detector seed (seed 43) on Colab

T4 GPU, *Runtime → Run all*, about 1.5 hours. This repeats the Phase 3 pipeline with one change: YOLOv8-m is retrained with seed 43 instead of 42. The recipe, thresholds and experiment configs are the committed ones; the seed is changed in a temporary copy of `configs/yolo_train.yaml`, so no file in the repository changes.

Nothing from Phase 3 is overwritten. Everything goes to new folders:

| What | Colab | Drive |
|---|---|---|
| detector | `runs/yolo/yolov8m_kvasir_s43/` | `MyDrive/F26-13/yolo/yolov8m_kvasir_s43/` |
| detection caches | `runs/detections_s43/` | `MyDrive/F26-13/detections_s43/` |
| runs B, T, C | `runs/s43/<run>/<dataset>/` | `MyDrive/F26-13/runs_phase5_s43/<run>/<dataset>/` |

Every step checks Drive first and reuses what is already there, so after a disconnect *Run all* only does what is missing (setup guide C8). The oracle run does not depend on the detector, so it is not repeated. Decision: DL-47.

## 1. GPU, Drive, code

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/proj
!git clone -q https://github.com/kashishfatima999/Confidence-Aware-Self-Prompting-for-Robust-Polyp-Segmentation.git /content/proj
%cd /content/proj
!git log -1 --format="commit: %h %s"

## 2. Packages (same pins as Phase 3, DL-26) and tests

In [ ]:
!pip install -q -r requirements-colab.txt
!SAM2_BUILD_CUDA=0 pip install -q "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4"
!pip freeze | grep -i -E "^torch==|^ultralytics==|SAM-2"
!python -m unittest discover -s tests 2>&1 | tail -3

## 3. Weights (reused from Drive)

In [ ]:
import os
W = "/content/drive/MyDrive/F26-13/weights"
os.makedirs(W, exist_ok=True); os.makedirs("weights", exist_ok=True)
!wget -q -nc -P {W} https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
!cd {W} && python -c "from ultralytics import YOLO; YOLO('yolov8m.pt')"
!ln -sf {W}/sam2_hiera_large.pt weights/sam2_hiera_large.pt
!ln -sf {W}/yolov8m.pt yolov8m.pt
!ls -la {W}

## 4. Data (same as Phase 3: Kaggle copies, our prepare scripts, committed split)

In [ ]:
import glob, json, shutil
import kagglehub
import pandas as pd
from PIL import Image
os.chdir("/content/proj")

p = kagglehub.dataset_download("debeshjha1/kvasirseg")
src = f"{p}/Kvasir-SEG/Kvasir-SEG"
dest = "data/raw/kvasir-seg/Kvasir-SEG"
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(f"{src}/images", f"{dest}/images")
shutil.copytree(f"{src}/masks", f"{dest}/masks")
boxes = {}
for csv_path in sorted(glob.glob(f"{src}/bbox/*.csv")):
    image_id = os.path.splitext(os.path.basename(csv_path))[0]
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    w, h = Image.open(f"{dest}/images/{image_id}.jpg").size
    boxes[image_id] = {"height": h, "width": w,
                       "bbox": [{"label": "polyp", "xmin": int(r["xmin"]), "ymin": int(r["ymin"]),
                                 "xmax": int(r["xmax"]), "ymax": int(r["ymax"])} for _, r in df.iterrows()]}
json.dump(boxes, open(f"{dest}/kavsir_bboxes.json", "w"))
print(len(os.listdir(f"{dest}/images")), "Kvasir-SEG images,", len(boxes), "box files")

In [ ]:
!python src/data/prepare_kvasir.py | head -3
p2 = kagglehub.dataset_download("balraj98/cvcclinicdb")
shutil.rmtree("data/raw/cvc-clinicdb", ignore_errors=True)
shutil.copytree(f"{p2}/PNG", "data/raw/cvc-clinicdb/PNG")
!python src/data/prepare_clinicdb.py | head -2

## 5. Train the detector with seed 43 (~25 min)

Same recipe as Phase 3 (DL-31) except `seed` and `name`. The full console output is saved as `train_log.txt` next to the weights, which Phase 3 could not keep (open item O2).

In [ ]:
import yaml
YOLO_DRIVE = "/content/drive/MyDrive/F26-13/yolo/yolov8m_kvasir_s43"
LOCAL = "runs/yolo/yolov8m_kvasir_s43"
recipe = yaml.safe_load(open("configs/yolo_train.yaml"))
assert recipe["seed"] == 42 and recipe["name"] == "yolov8m_kvasir_s42"   # the committed recipe, unchanged
recipe.update(seed=43, name="yolov8m_kvasir_s43")
yaml.safe_dump(recipe, open("/content/yolo_train_s43.yaml", "w"), sort_keys=False)
print(open("/content/yolo_train_s43.yaml").read())

if os.path.exists(f"{YOLO_DRIVE}/weights/best.pt"):
    print("reusing seed-43 weights from Drive")
    shutil.copytree(YOLO_DRIVE, LOCAL, dirs_exist_ok=True)
else:
    os.makedirs(LOCAL, exist_ok=True)
    !python src/train_yolo.py --recipe /content/yolo_train_s43.yaml 2>&1 | tee runs/yolo/yolov8m_kvasir_s43/train_log.txt | tail -40
    shutil.copytree(LOCAL, YOLO_DRIVE, dirs_exist_ok=True)
print(json.dumps(json.load(open(f"{LOCAL}/summary.json")), indent=2))
BEST = f"{LOCAL}/weights/best.pt"

## 6. Detection caches with the seed-43 detector (confidence 0.05, DL-33)

In [ ]:
DET_DRIVE = "/content/drive/MyDrive/F26-13/detections_s43"
os.makedirs(DET_DRIVE, exist_ok=True); os.makedirs("runs/detections_s43", exist_ok=True)
for ds in ["kvasir_test", "clinicdb"]:
    out = f"runs/detections_s43/{ds}.json"
    if os.path.exists(f"{DET_DRIVE}/{ds}.json"):
        shutil.copy(f"{DET_DRIVE}/{ds}.json", out); print(ds, "cache restored from Drive")
    else:
        !python src/pipeline/detect.py --weights {BEST} --dataset {ds} --out {out}
        shutil.copy(out, DET_DRIVE)
    e = json.load(open(out))["images"]
    print(f"{ds}: {len(e)} images, {sum(len(v['boxes']) for v in e.values())} candidate boxes")

## 7. Runs B, T, C on both test sets (~30 min)

The committed configs, with no `--set` overrides: B (`baseline.yaml`), T (`threshold_only.yaml`), C (`casp.yaml`, thresholds from DL-34). Each finished run is copied to Drive at once and is restored instead of recomputed after a reconnect.

In [ ]:
import subprocess, time
RUNS_DRIVE = "/content/drive/MyDrive/F26-13/runs_phase5_s43"
RUNS = {"baseline": "configs/experiments/baseline.yaml",
        "threshold_only": "configs/experiments/threshold_only.yaml",
        "casp": "configs/experiments/casp.yaml"}

def run_saved(name, ds):
    local, saved = f"runs/s43/{name}/{ds}", f"{RUNS_DRIVE}/{name}/{ds}"
    if os.path.exists(f"{saved}/summary.json"):
        shutil.copytree(saved, local, dirs_exist_ok=True)
        return json.load(open(f"{local}/summary.json")), "restored from Drive"
    t0 = time.time()
    cmd = (f"python src/pipeline/run_experiment.py --config {RUNS[name]} --dataset {ds} "
           f"--detections runs/detections_s43/{ds}.json --name s43/{name}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(f"{local}/summary.json"):
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(f"{name} on {ds} failed")
    shutil.copytree(local, saved, dirs_exist_ok=True)
    return json.load(open(f"{local}/summary.json")), f"done in {time.time() - t0:.0f} s"

for ds in ["kvasir_test", "clinicdb"]:
    for name in RUNS:
        s, status = run_saved(name, ds)
        print(f"{name:15s} {ds:12s} Dice {s['mean_dice']:.4f}  missed {s['images_with_empty_prediction']:3d}  "
              f"box P/R {s['precision']:.3f}/{s['recall']:.3f}   ({status})")

## 8. Check that everything is in Drive

In [ ]:
for d in [YOLO_DRIVE, DET_DRIVE, RUNS_DRIVE]:
    print(d)
    !find {d} -maxdepth 3 \( -name "*.json" -o -name "*.csv" -o -name "*.txt" \) | sort

## Next step (CPU, on a laptop)

Download `MyDrive/F26-13/runs_phase5_s43/` and put it next to the Phase 3 runs, then:

    python src/eval/seed_compare.py --s42 runs/runs_phase3 --s43 runs/runs_phase5_s43

The oracle run is taken from the seed-42 folder, since it does not use the detector.